In [4]:
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
from google.colab import auth
from oauth2client.client import GoogleCredentials

auth.authenticate_user()
gauth = GoogleAuth()
gauth.credentials = GoogleCredentials.get_application_default()
drive = GoogleDrive(gauth)

folder_id = '1d_l-uKZFXudiFpJqBKKkTmblM171meRW'

file_list = drive.ListFile({'q': f"'{folder_id}' in parents and trashed=false"}).GetList()

csv_file = None
for file in file_list:
    if file['title'] == 'L80-Short.pdf':
        pdf_file = file
        break

if pdf_file:
    print(f"Se descarcă: {pdf_file['title']}")
    pdf_file.GetContentFile('L80-Short.pdf')
else:
    print("Fișierul 'L80-Short.pdf' nu a fost găsit în folder.")

L80-Short.pdf
recenzii_100.csv
Se descarcă: L80-Short.pdf


In [29]:
open_ai_k = 'sk-proj-YvJ9AtlyXweWJLLlfP7nec-4bd83-_azWk2LEI7C4TQLO3EPscMcw9K_4Xm6EU7EWGcmQUDjqoT3BlbkFJBHAY9_HQ9-hcvTgSjCdQBO4RkLj3rfqkqDZodNHxMZxwA59GYKaG6WQchdu6nKlhamK'

In [30]:
open_ai_k += 'LastFewChars'

In [23]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 48.1 MB/s eta 0:00:00


In [24]:
import pdfplumber

In [33]:
def extrage_text_din_pdf(pdf_path, max_caractere=400000):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text() + "\n"
            if len(text) > max_caractere:
                break
    return text[:max_caractere]

In [34]:
document_text = extrage_text_din_pdf('L80-Short.pdf')

In [35]:
from openai import OpenAI
import os

client = OpenAI(api_key=open_ai_k)  # setează cheia ta

def raspunde_la_intrebare(document, intrebare):
    prompt = f"""
Primești un document sub formă de text și o întrebare.
Răspunde DOAR pe baza informației din document. Dacă informația nu este prezentă, răspunde: "Informația nu este prezentă în document".

Document:
\"\"\"
{document}
\"\"\"

Întrebare:
{intrebare}

Răspuns:
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Eroare: {e}"


In [ ]:
while True:
    intrebare = input(" Întrebare: ")
    if intrebare.lower() in ["quit", "exit"]:
        print("La revedere!")
        break
    raspuns = raspunde_la_intrebare(document_text, intrebare)
    print("Răspuns:\n", raspuns)
    print("-" * 60)

 Întrebare: Ce tipuri de infrastructuri sunt menționate ca exemple de active eligibile în țări terțe?
Răspuns:
 Informația nu este prezentă în document.
------------------------------------------------------------
 Întrebare: Ce valoare agregată aveau fondurile ELTIF în 2021?
Răspuns:
 Valoarea agregată a activelor nete ale fondurilor ELTIF a fost estimată la aproximativ 2 400 000 000 EUR în 2021.
------------------------------------------------------------
 Întrebare:  Ce active sunt excluse explicit din categoria de investiții eligibile?
Răspuns:
 Activele de investiții eligibile ar trebui înțelese ca excluzând operele de artă, manuscrisele, stocurile de vin, bijuteriile sau alte active, care nu reprezintă în sine investiții pe termen lung în economia reală.
------------------------------------------------------------
 Întrebare: Ce limite de capitalizare bursieră se aplică societăților eligibile listate la bursă?
Răspuns:
 Limita de capitalizare bursieră a societăților de portofoliu